# AEGIS — entraînement NER PII sur **Google Colab**

Pipeline aligné sur [`training/README.md`](https://github.com/zokastech/aegis/blob/master/training/README.md) : données synthétiques EU (IOB2), fine-tuning **XLM-RoBERTa-base**, **évaluation + comparaison Presidio** (F2 sur le même jeu gold), export **ONNX** pour `aegis-ner`.

**Runtime recommandé : GPU Tesla T4** (~16 Go VRAM, offert sur Colab Pro / selon quota). Menu *Exécution* → *Modifier le type d'exécution* → **T4 GPU**. Les hyperparamètres d’entraînement (batch 8, `max_seq_length` 512, fp16, gradient checkpointing) sont calibrés pour ce profil. **L4 / A100** : vous pouvez augmenter le batch ; en **CPU** le notebook bascule seul sur des réglages plus légers.

**Sécurité** : n’utilisez que des données synthétiques ou anonymisées ; ne collez pas de vraies PII.

Licence : Apache-2.0 / MIT — [zokastech.fr](https://zokastech.fr).

## 1. GPU et clone du dépôt

Vérifiez d’abord la présence d’un **GPU NVIDIA** (`nvidia-smi`). Après l’installation des dépendances, la cellule *Chemins* confirme le nom du GPU (ex. **Tesla T4**).

Par défaut le notebook clone `zokastech/aegis` dans `/content/aegis`. Pour une autre branche : définissez `BRANCH` ci-dessous.

In [ ]:
# @title Vérifier le GPU
!nvidia-smi 2>/dev/null || echo "Pas de GPU NVIDIA — passez en runtime GPU ou l'entraînement sera très lent sur CPU."

In [ ]:
# @title Cloner Zokastech AEGIS (branche par défaut : master)
import shutil
import subprocess
from pathlib import Path

BRANCH = "master"  # @param {type:"string"}  upstream Zokastech/AEGIS utilise « master » ; « main » est essayé en secours si besoin
REPO_URL = "https://github.com/zokastech/aegis.git"
ROOT = Path("/content")
REPO_ROOT = ROOT / "aegis"


def _git(args, **kw):
    print("+", " ".join(args))
    return subprocess.run(args, **kw)


if REPO_ROOT.is_dir():
    _git(["git", "-C", str(REPO_ROOT), "fetch", "origin"], check=False)
    _git(["git", "-C", str(REPO_ROOT), "checkout", BRANCH], check=True)
    _git(["git", "-C", str(REPO_ROOT), "pull"], check=False)
else:
    last_err = ""
    for b in [BRANCH, "master", "main"]:
        if REPO_ROOT.exists():
            shutil.rmtree(REPO_ROOT)
        r = _git(
            ["git", "clone", "--depth", "1", "--branch", b, str(REPO_URL), str(REPO_ROOT)],
            capture_output=True,
            text=True,
        )
        if r.returncode == 0:
            print("Cloné — branche utilisée :", b)
            break
        last_err = (r.stderr or r.stdout or "").strip()
    else:
        raise RuntimeError(
            f"git clone a échoué (essayé {BRANCH!r}, master, main). Dernière erreur :\n{last_err}"
        )

%cd {REPO_ROOT}
assert (REPO_ROOT / "training" / "dataset_builder.py").is_file(), "Clone incomplet : training/dataset_builder.py manquant"
print("REPO_ROOT =", REPO_ROOT.resolve())

In [ ]:
# @title Dépendances Python (training)
import subprocess
import sys

# Colab : NumPy 2.x ou install cassée → « dtype size changed » à l’import (datasets/pandas/pyarrow).
# training/requirements.txt pinne numpy<2 ; on désinstalle puis réaligne les wheels C.
_NUMPY = "numpy>=1.26.4,<2.0"
req = REPO_ROOT / "training" / "requirements.txt"

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U", "pip"])
subprocess.run(
    [sys.executable, "-m", "pip", "uninstall", "-y", "numpy"],
    capture_output=True,
)
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", _NUMPY])
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", str(req)])
subprocess.check_call(
    [sys.executable, "-m", "pip", "install", "-q", "--force-reinstall", "--no-deps", _NUMPY]
)
# Paquets avec extensions C liées à NumPy (éviter mélange d’ABI après -r).
subprocess.check_call(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "--force-reinstall",
        _NUMPY,
        "pandas",
        "pyarrow",
        "scikit-learn",
    ]
)

import numpy as _np

print("OK —", sys.executable, "| NumPy", _np.__version__)
print(
    "Si une cellule suivante échoue encore sur NumPy : menu Exécution → Redémarrer la session, "
    "puis réexécutez Clone → Dépendances → Chemins (sans sauter)."
)

In [ ]:
# @title Chemins, PYTHONPATH et préréglages **T4** (VRAM)
import os
import sys
from pathlib import Path

# Colab + PyTorch 2.x : limite la fragmentation sur GPU 16 Go (T4).
if os.environ.get("COLAB_RELEASE_TAG"):
    os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

TRAINING_DIR = REPO_ROOT / "training"
DATA_DIR = TRAINING_DIR / "data"
OUTPUT_DIR = TRAINING_DIR / "outputs"
EXPORT_DIR = TRAINING_DIR / "exports" / "onnx_colab"

os.chdir(REPO_ROOT)
sys.path.insert(0, str(TRAINING_DIR))

DATA_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

try:
    import torch

    if torch.cuda.is_available():
        p = torch.cuda.get_device_properties(0)
        print(
            "CUDA:",
            torch.cuda.get_device_name(0),
            "|",
            f"{p.total_memory / (1024**3):.1f} Go VRAM",
            "| capability",
            ".".join(map(str, torch.cuda.get_device_capability(0))),
        )
    else:
        print("CUDA indisponible — entraînement CPU prévu (section 4).")
except ImportError:
    print("PyTorch non importable — réexécutez la cellule « Dépendances ».")

print("TRAINING_DIR =", TRAINING_DIR)

## 2. Jeu synthétique (IOB2, multilingue UE)

`NUM_SYNTHETIC` petit = démo rapide. Pour viser une meilleure qualité, montez à **50_000** (temps + mémoire plus élevés).

Si vous voyez une erreur **NumPy / dtype size changed** : **redémarrez la session** Colab, puis enchaînez **Clone → Dépendances → Chemins** avant cette cellule (évite un NumPy déjà chargé en mémoire).

In [ ]:
from dataset_builder import build_dataset

NUM_SYNTHETIC = 5000  # @param {type:"integer"}
SEED = 42
SYNTH_PATH = DATA_DIR / "colab_eu_pii_synthetic"

SYNTH_PATH.parent.mkdir(parents=True, exist_ok=True)
ds_synth = build_dataset(NUM_SYNTHETIC, seed=SEED)
ds_synth.save_to_disk(str(SYNTH_PATH))
print(ds_synth)
print("Sauvegardé:", SYNTH_PATH)

## 3. Fusion avec des JSONL du dépôt (optionnel)

Si les fichiers d’exemple existent dans le clone, on fusionne avec le synthétique. Sinon on entraîne **uniquement** sur le jeu synthétique.

In [ ]:
import subprocess
import sys

CUSTOM_JSONL = [
    REPO_ROOT / "datasets" / "training" / "ner_custom" / "samples.jsonl",
    REPO_ROOT / "datasets" / "training" / "ner_financial_seed" / "samples.jsonl",
]
existing = [p for p in CUSTOM_JSONL if p.is_file()]
CUSTOM_HF_PATH = DATA_DIR / "colab_from_jsonl"
MERGED_PATH = DATA_DIR / "colab_merged_synth_custom"
DATASET_FOR_TRAIN = SYNTH_PATH

if existing:
    subprocess.check_call(
        [
            sys.executable,
            str(TRAINING_DIR / "jsonl_to_hf_dataset.py"),
            *[str(p) for p in existing],
            "--output",
            str(CUSTOM_HF_PATH),
            "--val_ratio",
            "0.2",
            "--seed",
            str(SEED),
        ],
        cwd=str(TRAINING_DIR),
    )
    subprocess.check_call(
        [
            sys.executable,
            str(TRAINING_DIR / "merge_hf_datasets.py"),
            str(SYNTH_PATH),
            str(CUSTOM_HF_PATH),
            "--output",
            str(MERGED_PATH),
        ],
        cwd=str(TRAINING_DIR),
    )
    DATASET_FOR_TRAIN = MERGED_PATH
    print("Jeu fusionné:", MERGED_PATH)
else:
    print("Aucun JSONL d’exemple — entraînement sur synthétique seul:", SYNTH_PATH)

## 4. Fine-tuning (`train_ner.py`) — profil **T4**

Sur **CUDA** (T4) : **fp16** + **gradient checkpointing** + défauts *batch 8*, *`max_seq_length` 512*, *accumulation 1* (modifiables ci-dessous). Les variables d’environnement `AEGIS_TRAIN_BATCH`, `AEGIS_TRAIN_GRAD_ACCUM`, `AEGIS_MAX_SEQ_LENGTH` priment sur ces champs si vous les définissez.

**OOM sur T4** : passez *batch* à **6** ou **4**, ou *max_seq* à **384** / **256**, ou augmentez *grad_accum* pour garder un batch effectif.

En **CPU** : batch réduit, pas de fp16.

In [ ]:
import importlib.util
import os
import subprocess
import sys

for _pkg in ("seqeval", "accelerate"):
    if importlib.util.find_spec(_pkg) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", _pkg])

import torch

MODEL_OUT = OUTPUT_DIR / "ner-colab-xlmr"
MODEL_OUT.mkdir(parents=True, exist_ok=True)

use_cuda = torch.cuda.is_available()
use_cpu_flag = os.environ.get("AEGIS_TRAIN_CPU", "").lower() in ("1", "true", "yes") or not use_cuda

# Défauts calibrés **Tesla T4** (~16 Go) + XLM-R-base + checkpointing ; @param = réglages Colab.
T4_BATCH = 8  # @param {type:"integer"}
T4_GRAD_ACCUM = 1  # @param {type:"integer"}
T4_MAX_SEQ = 512  # @param {type:"integer"}

if use_cuda:
    BATCH = int(os.environ.get("AEGIS_TRAIN_BATCH", str(T4_BATCH)))
    GRAD_ACCUM = int(os.environ.get("AEGIS_TRAIN_GRAD_ACCUM", str(T4_GRAD_ACCUM)))
    MAX_SEQ = int(os.environ.get("AEGIS_MAX_SEQ_LENGTH", str(T4_MAX_SEQ)))
    FP16 = True
    torch.cuda.empty_cache()
else:
    BATCH = int(os.environ.get("AEGIS_TRAIN_BATCH", "4"))
    GRAD_ACCUM = int(os.environ.get("AEGIS_TRAIN_GRAD_ACCUM", "2"))
    MAX_SEQ = int(os.environ.get("AEGIS_MAX_SEQ_LENGTH", "256"))
    FP16 = False

NUM_EPOCHS = 2  # @param {type:"integer"}

train_cmd = [
    sys.executable,
    str(TRAINING_DIR / "train_ner.py"),
    "--dataset",
    str(DATASET_FOR_TRAIN),
    "--output_dir",
    str(MODEL_OUT),
    "--num_train_epochs",
    str(NUM_EPOCHS),
    "--per_device_train_batch_size",
    str(BATCH),
    "--per_device_eval_batch_size",
    str(min(BATCH, 8)),
    "--gradient_accumulation_steps",
    str(GRAD_ACCUM),
    "--max_seq_length",
    str(MAX_SEQ),
    "--gradient_checkpointing",
]
if use_cpu_flag:
    train_cmd.append("--cpu")
if FP16 and not use_cpu_flag:
    train_cmd.append("--fp16")

_train_env = os.environ.copy()
if use_cuda and not use_cpu_flag:
    _train_env.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

_dev = (
    "CUDA + fp16 (T4)"
    if (use_cuda and FP16 and "T4" in torch.cuda.get_device_name(0))
    else "CUDA + fp16"
    if (use_cuda and FP16)
    else "CPU"
    if use_cpu_flag
    else "MPS/other"
)
print(
    "Device:",
    _dev,
    "| batch",
    BATCH,
    "| grad_accum",
    GRAD_ACCUM,
    "| max_seq",
    MAX_SEQ,
)
print(" ".join(train_cmd))
subprocess.check_call(train_cmd, cwd=str(TRAINING_DIR), env=_train_env)

## 5. Test inférence + évaluation (F2) et **Presidio**

- **Test rapide** : quelques phrases tokenisées → étiquettes IOB2 du checkpoint `best_hf`.
- **`evaluate.py`** : même protocole que la doc `training/README.md` — F1/F2 par type, matrice de confusion, rapport HTML. Avec **`--with_presidio`**, Presidio sert de **baseline** sur le split *validation* (alignement token ↔ spans). Les scores Presidio dépendent de **`--presidio_language`** (`fr` ou `en` : installez le modèle spaCy dans la cellule *Évaluation* (après le test rapide)).

**Indicateur « expertise »** : comparez le **F2 micro (entités)** du modèle à celui de Presidio sur le même jeu gold ; un écart positif (modèle − Presidio) indique que le modèle dépasse la baseline sur cette métrique.

*Le jeu synthétique est multilingue ; une seule langue Presidio pour tout le split est une approximation.*

In [ ]:
# @title Inférence rapide (aperçu après entraînement)
import importlib.util

from transformers import AutoModelForTokenClassification

from hf_tokenizer_utils import load_autotokenizer_pretrained
from ner_span_iob import tags_to_spans

_ev_path = TRAINING_DIR / "evaluate.py"
_spec = importlib.util.spec_from_file_location("aegis_ner_evaluate", _ev_path)
_ev = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(_ev)

BEST_HF = MODEL_OUT / "best_hf"
tok = load_autotokenizer_pretrained(str(BEST_HF), use_fast=True)
model = AutoModelForTokenClassification.from_pretrained(BEST_HF)
model.eval()

examples = [
    ["Contact", ":", "jean.dupont@mail.fr", "—", "tel", ".", "0612345678"],
    ["IBAN", ":", "FR1420041010050500013M02606"],
    ["M.", "Martin", "a", "payé", "avec", "la", "carte", "4532015112830366"],
]

for tokens in examples:
    tags = _ev.refine_iob(_ev.predict_tags_per_word(model, tok, tokens))
    spans = tags_to_spans(tags)
    print("Tokens:", tokens)
    print("Tags  :", tags)
    print("Spans :", [(s.etype, s.start, s.end) for s in spans])
    print("---")

In [ ]:
# @title Modèles spaCy (Presidio) + rapport d’évaluation
import subprocess
import sys

from IPython.display import HTML, display

BEST_HF = MODEL_OUT / "best_hf"
REPORTS_DIR = TRAINING_DIR / "reports"
REPORTS_DIR.mkdir(parents=True, exist_ok=True)
EVAL_REPORT = REPORTS_DIR / "ner_eval_colab_presidio.html"

PRESIDIO_LANG = "fr"  # @param ["fr", "en"]
MAX_EVAL_SAMPLES = 1500  # @param {type:"integer"}
RUN_WITH_PRESIDIO = True  # @param {type:"boolean"}

if not BEST_HF.is_dir() or not (BEST_HF / "config.json").is_file():
    raise FileNotFoundError(f"Entraînez d’abord le modèle (section 4). Manquant : {BEST_HF}")

if RUN_WITH_PRESIDIO:
    _spacy_models = {"fr": "fr_core_news_sm", "en": "en_core_web_sm"}
    _sm = _spacy_models.get(PRESIDIO_LANG, "en_core_web_sm")
    subprocess.check_call(
        [sys.executable, "-m", "spacy", "download", _sm],
        cwd=str(TRAINING_DIR),
    )

eval_cmd = [
    sys.executable,
    str(TRAINING_DIR / "evaluate.py"),
    "--dataset",
    str(DATASET_FOR_TRAIN),
    "--model_dir",
    str(BEST_HF),
    "--split",
    "validation",
    "--max_samples",
    str(MAX_EVAL_SAMPLES),
    "--out_report",
    str(EVAL_REPORT),
    "--presidio_language",
    PRESIDIO_LANG,
]
if RUN_WITH_PRESIDIO:
    eval_cmd.append("--with_presidio")

print(" ".join(eval_cmd))
subprocess.check_call(eval_cmd, cwd=str(TRAINING_DIR))

if EVAL_REPORT.is_file():
    display(HTML(EVAL_REPORT.read_text(encoding="utf-8")))
    print("Fichier rapport :", EVAL_REPORT)

## 6. Hugging Face Hub (optionnel)

1. Créez un token sur [huggingface.co/settings/tokens](https://huggingface.co/settings/tokens).
2. Dans Colab : *Secrets* (icône clé) → `HF_TOKEN`, ou `huggingface-cli login` dans une cellule.
3. Renseignez `HF_REPO_ID` (ex. `votre_org/aegis-ner-eu-pii`).

In [ ]:
import os
import subprocess
import sys

HF_REPO_ID = ""  # @param {type:"string"}
HF_PRIVATE = False  # @param {type:"boolean"}

if not HF_REPO_ID.strip():
    HF_REPO_ID = os.environ.get("AEGIS_HF_REPO_ID", "").strip()

BEST_HF = MODEL_OUT / "best_hf"
if not HF_REPO_ID:
    print("Aucun HF_REPO_ID — étape ignorée.")
elif not BEST_HF.is_dir() or not (BEST_HF / "config.json").is_file():
    print("best_hf introuvable — lancez d’abord l’entraînement (section 4).")
else:
    push_cmd = [
        sys.executable,
        str(TRAINING_DIR / "push_hf_model.py"),
        "--model_dir",
        str(BEST_HF),
        "--repo_id",
        HF_REPO_ID,
    ]
    if HF_PRIVATE:
        push_cmd.append("--private")
    subprocess.check_call(push_cmd, cwd=str(TRAINING_DIR))
    print("Hub:", f"https://huggingface.co/{HF_REPO_ID}")

## 7. Export ONNX (`aegis-ner`)

Sortie dans `training/exports/onnx_colab/` : `model.onnx`, `model_int8.onnx`, `tokenizer_hf/tokenizer.json`.

In [ ]:
import subprocess
import sys

BEST_HF = MODEL_OUT / "best_hf"
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

if not BEST_HF.is_dir():
    raise FileNotFoundError(f"Checkpoint manquant: {BEST_HF}")

export_cmd = [
    sys.executable,
    str(TRAINING_DIR / "export_onnx.py"),
    "--model_dir",
    str(BEST_HF),
    "--out_dir",
    str(EXPORT_DIR),
]
print(" ".join(export_cmd))
subprocess.check_call(export_cmd, cwd=str(TRAINING_DIR))
print("Export terminé:", EXPORT_DIR)

## 8. Sauvegarder sur Google Drive (optionnel)

Les fichiers sous `/content` sont **perdus** quand la session Colab se termine. Montez Drive et copiez `best_hf`, l’export ONNX et, si présent, le rapport **`ner_eval_colab_presidio.html`**.

In [ ]:
# @title Monter Drive et copier les artefacts
from pathlib import Path

SAVE_TO_DRIVE = False  # @param {type:"boolean"}
DRIVE_SUBFOLDER = "AEGIS_NER_colab"  # @param {type:"string"}

if not SAVE_TO_DRIVE:
    print("SAVE_TO_DRIVE=False — rien à copier.")
else:
    from google.colab import drive
    import shutil

    drive.mount("/content/drive")
    dest_root = Path("/content/drive/MyDrive") / DRIVE_SUBFOLDER
    dest_root.mkdir(parents=True, exist_ok=True)

    if MODEL_OUT.joinpath("best_hf").is_dir():
        shutil.copytree(MODEL_OUT / "best_hf", dest_root / "best_hf", dirs_exist_ok=True)
    if EXPORT_DIR.is_dir():
        shutil.copytree(EXPORT_DIR, dest_root / "onnx_export", dirs_exist_ok=True)
    _rep = TRAINING_DIR / "reports" / "ner_eval_colab_presidio.html"
    if _rep.is_file():
        shutil.copy2(_rep, dest_root / _rep.name)
    print("Copié vers:", dest_root)

## Références

- [`training/README.md`](https://github.com/zokastech/aegis/blob/master/training/README.md) — évaluation : `evaluate.py`, option `--with_presidio`
- Notebook local : [`train_ner_pii.ipynb`](train_ner_pii.ipynb)
- Jeux HF publics : [`train_ner_hf_public.ipynb`](train_ner_hf_public.ipynb)
- Exemple CLI : `python training/evaluate.py --dataset …/data/... --model_dir …/best_hf --with_presidio --presidio_language fr --out_report report.html`
